In [ ]:
import numpy as np
import pandas as pd
import config as cfg
from matplotlib import pyplot as plt

plt.style.use("default")
plt.rcParams.update({
    "axes.labelsize": "large",
    "axes.titlesize": "large",
    "font.size": 16.0,
    "legend.fontsize": "medium",
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsfonts} \usepackage{amstext} \usepackage{bm}"
})

# --- Load CG-Q ablation results ---
filename = f"{cfg.RESULTS_DIR}/synthetic_results_bguw_vs_cgq.csv"
df = pd.read_csv(filename)

print(df.columns)
print(df.head())

# Expected columns:
# copula_name, k_tau, err_bguw, err_cgq
# If your file uses slightly different names, adjust here.
metric1, metric2 = "err_bguw", "err_cgq"
label1, label2 = r"IBS-Dep-UW", r"IBS-CG-Q"

# Fallbacks if the script saved different names
if metric1 not in df.columns:
    metric1 = "err_dep_bguw"
if metric2 not in df.columns:
    metric2 = "err_dep_cgq"

sigma_level = 1.0

# Average over copula families within seed to keep the plot small
seed_level = (
    df.groupby(["k_tau", "seed"], as_index=False)
      .agg(
          bguw=(metric1, "mean"),
          cgq=(metric2, "mean"),
      )
)

grouped = (
    seed_level.groupby("k_tau", as_index=False)
              .agg(
                  bguw_mean=("bguw", "mean"),
                  bguw_std=("bguw", "std"),
                  cgq_mean=("cgq", "mean"),
                  cgq_std=("cgq", "std"),
              )
              .sort_values("k_tau")
)

# --- Compact plot: about 25% of one column page ---
fig, ax = plt.subplots(1, 1, figsize=(3.35, 1.85))

x = grouped["k_tau"].to_numpy()

m1 = grouped["bguw_mean"].to_numpy()
s1 = grouped["bguw_std"].to_numpy()

m2 = grouped["cgq_mean"].to_numpy()
s2 = grouped["cgq_std"].to_numpy()

ax.plot(x, m1, linestyle="dashed", marker="*", markersize=6,
        linewidth=2.2, alpha=0.85, label=label1)
ax.plot(x, m2, linestyle="dashed", marker="o", markersize=5,
        linewidth=2.2, alpha=0.85, label=label2)

if np.any(~np.isnan(s1)):
    ax.fill_between(x, m1 - sigma_level * s1, m1 + sigma_level * s1, alpha=0.2)
if np.any(~np.isnan(s2)):
    ax.fill_between(x, m2 - sigma_level * s2, m2 + sigma_level * s2, alpha=0.2)

ax.set_ylabel(r"Error", fontsize=13)
ax.set_xlabel(r"Kendall's $\tau$", fontsize=13)

ax.set_xticks([0.0, 0.25, 0.5, 0.75])
ax.set_xticklabels(["0.00", "0.25", "0.50", "0.75"], fontsize=11)

ax.tick_params(axis="y", labelsize=11, length=0)
ax.tick_params(axis="x", length=0)

ax.grid(True, alpha=0.5)
ax.legend(loc="upper left", fontsize=9, frameon=True)

plt.tight_layout()
plt.savefig(
    f"{cfg.PLOTS_DIR}/synthetic_bguw_vs_cgq_small.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.show()